# Setup — your inventory data + Fabric Data Agent (run once)

Run this notebook **inside your own Fabric workspace** — the one you created and assigned to your F2 capacity (see Challenge 1).

**Prerequisites**
1. Your **F2 capacity** (`FabricCapacityName` from your lab dashboard) is **running**, and this workspace is assigned to it.
2. A Lakehouse named **`InventoryLakehouse`** exists in this workspace and is attached to this notebook (Explorer → **Add Lakehouse**).
3. That's it — the seed data is **embedded in this notebook**, so there is no upload or download.

This notebook: loads the embedded seed data → builds the `Inventory` and `DemandHistory` fact tables deterministically → writes 7 Lakehouse tables → creates and **publishes** your `inventory-hack-agent` Fabric Data Agent → prints **your** Workspace ID and Agent ID for Challenge 2.


In [ ]:
# Install both preview SDKs here, at the very top. %pip RESTARTS the Spark kernel and
# wipes every variable defined before it — so ALL installs must happen up front, and
# there must be NO later %pip cell (a second restart would wipe `agent`, etc.).
%pip install -q fabric-data-agent-sdk mcp


In [ ]:
import json, math
from datetime import date, timedelta

# ---------------------------------------------------------------------------
# Seed data is embedded inline so this notebook is fully self-contained — no file
# upload and no external download, regardless of where this repo ultimately lives.
# This notebook is the single source of truth for the seed data.
# ---------------------------------------------------------------------------
products = [
    {"productId": "P001", "name": "Lawn Mower Pro",     "category": "garden_and_lawn",     "unitCost": 180, "leadTimeDays": 14, "supplierId": "S01", "baseWeeklyUnits": 8,  "baseOnHand": 40},
    {"productId": "P002", "name": "Garden Hose 50ft",   "category": "garden_and_lawn",     "unitCost": 25,  "leadTimeDays": 7,  "supplierId": "S01", "baseWeeklyUnits": 30, "baseOnHand": 150},
    {"productId": "P003", "name": "Fertilizer 20lb",    "category": "garden_and_lawn",     "unitCost": 18,  "leadTimeDays": 10, "supplierId": "S01", "baseWeeklyUnits": 45, "baseOnHand": 200},
    {"productId": "P004", "name": "Leaf Blower X2",      "category": "outdoor_power_tools", "unitCost": 95,  "leadTimeDays": 21, "supplierId": "S02", "baseWeeklyUnits": 20, "baseOnHand": 30},
    {"productId": "P005", "name": "Chainsaw 16in",       "category": "outdoor_power_tools", "unitCost": 140, "leadTimeDays": 21, "supplierId": "S02", "baseWeeklyUnits": 12, "baseOnHand": 25},
    {"productId": "P006", "name": "Hedge Trimmer",       "category": "outdoor_power_tools", "unitCost": 85,  "leadTimeDays": 18, "supplierId": "S02", "baseWeeklyUnits": 15, "baseOnHand": 28},
    {"productId": "P007", "name": "Interior Paint 1gal", "category": "paint_and_supplies",  "unitCost": 22,  "leadTimeDays": 7,  "supplierId": "S03", "baseWeeklyUnits": 50, "baseOnHand": 300},
    {"productId": "P008", "name": "Paint Roller Set",    "category": "paint_and_supplies",  "unitCost": 12,  "leadTimeDays": 5,  "supplierId": "S03", "baseWeeklyUnits": 40, "baseOnHand": 250},
    {"productId": "P009", "name": "Painters Tape",       "category": "paint_and_supplies",  "unitCost": 6,   "leadTimeDays": 5,  "supplierId": "S03", "baseWeeklyUnits": 60, "baseOnHand": 400},
    {"productId": "P010", "name": "Smart Thermostat",    "category": "smart_home",          "unitCost": 110, "leadTimeDays": 12, "supplierId": "S04", "baseWeeklyUnits": 18, "baseOnHand": 90},
    {"productId": "P011", "name": "Video Doorbell",      "category": "smart_home",          "unitCost": 130, "leadTimeDays": 12, "supplierId": "S04", "baseWeeklyUnits": 16, "baseOnHand": 80},
    {"productId": "P012", "name": "Smart Plug 4-pack",   "category": "smart_home",          "unitCost": 28,  "leadTimeDays": 9,  "supplierId": "S04", "baseWeeklyUnits": 35, "baseOnHand": 180},
]

suppliers = [
    {"supplierId": "S01", "name": "GreenField Supply", "region": "Midwest",   "reliabilityScore": 0.92},
    {"supplierId": "S02", "name": "PowerTool Depot",   "region": "West",      "reliabilityScore": 0.85},
    {"supplierId": "S03", "name": "ColorCraft Paints", "region": "Northeast", "reliabilityScore": 0.95},
    {"supplierId": "S04", "name": "ConnectHome",       "region": "West",      "reliabilityScore": 0.88},
]

stores = [
    {"storeId": "ST01", "name": "Boston Store",      "type": "retail",    "region": "Northeast",         "city": "Boston",   "state": "MA"},
    {"storeId": "ST02", "name": "Brooklyn Store",    "type": "retail",    "region": "Northeast",         "city": "Brooklyn", "state": "NY"},
    {"storeId": "ST03", "name": "Portland Store",    "type": "retail",    "region": "Pacific Northwest", "city": "Portland", "state": "OR"},
    {"storeId": "ST04", "name": "Seattle Store",     "type": "retail",    "region": "Pacific Northwest", "city": "Seattle",  "state": "WA"},
    {"storeId": "WH01", "name": "Chicago Warehouse", "type": "warehouse", "region": "Midwest",           "city": "Chicago",  "state": "IL"},
    {"storeId": "WH02", "name": "Dallas Warehouse",  "type": "warehouse", "region": "South",             "city": "Dallas",   "state": "TX"},
]

signals = [
    {"signalId": "SIG01", "signalType": "weather",      "region": "Pacific Northwest", "headline": "Prolonged heatwave and early spring across the Pacific Northwest", "score": 0.80, "source": "WeatherIQ", "affectedCategories": "outdoor_power_tools,garden_and_lawn"},
    {"signalId": "SIG02", "signalType": "search_trend", "region": "National",          "headline": "Sharp spike in online searches for leaf blowers and lawn tools", "score": 0.75, "source": "SearchIQ", "affectedCategories": "outdoor_power_tools"},
    {"signalId": "SIG03", "signalType": "competitor",   "region": "West",              "headline": "Major competitor reported out of stock on chainsaws", "score": 0.60, "source": "MarketIQ", "affectedCategories": "outdoor_power_tools"},
    {"signalId": "SIG04", "signalType": "news",         "region": "National",          "headline": "Home improvement season demand surge expected", "score": 0.50, "source": "NewsIQ", "affectedCategories": "paint_and_supplies,garden_and_lawn"},
    {"signalId": "SIG05", "signalType": "promo",        "region": "National",          "headline": "Competitor launches smart-home bundle promotion", "score": 0.40, "source": "MarketIQ", "affectedCategories": "smart_home"},
    {"signalId": "SIG06", "signalType": "weather",      "region": "Northeast",         "headline": "Active storm season forecast for the Northeast", "score": 0.55, "source": "WeatherIQ", "affectedCategories": "outdoor_power_tools"},
    {"signalId": "SIG07", "signalType": "supply",       "region": "West",              "headline": "Supplier delay reported at PowerTool Depot", "score": 0.65, "source": "SupplyIQ", "affectedCategories": "outdoor_power_tools"},
    {"signalId": "SIG08", "signalType": "search_trend", "region": "National",          "headline": "Interior painting projects trending on social media", "score": 0.45, "source": "SearchIQ", "affectedCategories": "paint_and_supplies"},
]

orders_in = [
    {"orderId": "ORD-1001", "type": "purchase", "status": "submitted", "requestedBy": "Inventory Optimisation Agent", "approvedBy": "Sam Rivera", "createdDate": "2026-06-15", "items": [{"sku": "P007", "location": "WH01", "quantity": 200}]},
    {"orderId": "ORD-1002", "type": "transfer", "status": "completed", "requestedBy": "Planning Team",                "approvedBy": "Sam Rivera", "createdDate": "2026-06-22", "items": [{"sku": "P002", "location": "ST01", "quantity": 60}]},
    {"orderId": "ORD-1003", "type": "purchase", "status": "rejected",  "requestedBy": "Inventory Optimisation Agent", "approvedBy": "Sam Rivera", "createdDate": "2026-06-28", "items": [{"sku": "P010", "location": "WH02", "quantity": 120}]},
    {"orderId": "ORD-1004", "type": "purchase", "status": "submitted", "requestedBy": "Inventory Optimisation Agent", "approvedBy": "Jordan Lee", "createdDate": "2026-07-02", "items": [{"sku": "P004", "location": "WH01", "quantity": 150}]},
]

_overrides = [
    {"productId": "P004", "storeId": "ST03", "onHand": 8},
    {"productId": "P004", "storeId": "ST04", "onHand": 6},
    {"productId": "P004", "storeId": "WH01", "onHand": 45},
    {"productId": "P004", "storeId": "WH02", "onHand": 40},
    {"productId": "P005", "storeId": "ST03", "onHand": 5},
    {"productId": "P005", "storeId": "ST04", "onHand": 7},
    {"productId": "P006", "storeId": "ST03", "onHand": 12},
]
overrides = {(o["productId"], o["storeId"]): o["onHand"] for o in _overrides}

print(f"Loaded {len(products)} products, {len(stores)} locations, {len(signals)} signals.")


In [ ]:
# ---- Build Inventory (12 products x 6 locations = 72 rows), deterministic ----
# reorderPoint = ceil(weekly demand over the supplier lead time)
# safetyStock  = ceil(reorderPoint * 0.5)
# warehouses hold ~4x a retail location's base stock; story SKUs come from overrides.
inventory = []
for p in products:
    rp = math.ceil(p["baseWeeklyUnits"] * p["leadTimeDays"] / 7)
    ss = math.ceil(rp * 0.5)
    for s in stores:
        on_hand = p["baseOnHand"] * 4 if s["type"] == "warehouse" else p["baseOnHand"]
        on_hand = overrides.get((p["productId"], s["storeId"]), on_hand)
        inventory.append({
            "productId": p["productId"], "storeId": s["storeId"],
            "onHand": int(on_hand), "reorderPoint": int(rp), "safetyStock": int(ss)
        })

# ---- Build DemandHistory: last 6 weeks, retail stores, +40% recent uplift for tools ----
WEEKS = 6
anchor_monday = date(2026, 7, 6)
retail = [s for s in stores if s["type"] == "retail"]
demand = []
for p in products:
    per_store = max(1, round(p["baseWeeklyUnits"] / len(retail)))
    for w in range(WEEKS):
        wk = anchor_monday - timedelta(weeks=(WEEKS - 1 - w))
        for s in retail:
            units = per_store
            if p["category"] == "outdoor_power_tools" and w >= WEEKS - 2:
                units = round(units * 1.4)  # recent demand spike
            demand.append({"productId": p["productId"], "storeId": s["storeId"],
                            "weekStart": wk.isoformat(), "units": int(units)})

# ---- Flatten replenishment orders (items -> itemsJson string) ----
orders = []
for o in orders_in:
    orders.append({
        "orderId": o["orderId"], "type": o["type"], "status": o["status"],
        "requestedBy": o["requestedBy"], "approvedBy": o["approvedBy"],
        "createdDate": o["createdDate"], "itemsJson": json.dumps(o["items"])
    })

print(f"Inventory rows: {len(inventory)} | DemandHistory rows: {len(demand)} | Orders: {len(orders)}")

In [ ]:
# A default Lakehouse must be attached, or saveAsTable fails with a cryptic
# "No default context found" Spark error. Check early and explain the fix.
import notebookutils
if not notebookutils.runtime.context.get("defaultLakehouseId"):
    raise RuntimeError(
        "No default Lakehouse is attached to this notebook. In the Explorer pane (left), "
        "click Add > Existing Lakehouse > InventoryLakehouse (in your inventory-hack workspace), "
        "then Run all again."
    )

# ---- Write all 7 tables to the Lakehouse (Delta) ----
def write_table(name, rows):
    spark.createDataFrame(rows).write.mode("overwrite").format("delta").saveAsTable(name)
    print(f"  wrote {name} ({len(rows)} rows)")

write_table("Products", products)
write_table("Suppliers", suppliers)
write_table("Stores", stores)
write_table("Inventory", inventory)
write_table("DemandHistory", demand)
write_table("ExternalSignals", signals)
write_table("ReplenishmentOrders", orders)
print("All tables written.")

## Create + publish the Fabric Data Agent

> **Preview API note:** `fabric-data-agent-sdk` is in preview and method names can change between versions. If a call below fails, check the [SDK reference](https://learn.microsoft.com/fabric/data-science/fabric-data-agent-sdk) and adjust. After publishing, the last cell prints your **Workspace ID** and **Data Agent (artifact) ID** — note both; you'll paste them into the Fabric Data Agent tool in **Challenge 2**.


In [ ]:
import time
import notebookutils
from fabric.dataagent.client import create_data_agent
try:
    from fabric.dataagent.client import FabricDataAgentManagement
except Exception:
    FabricDataAgentManagement = None

workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
lakehouse_id = notebookutils.runtime.context.get("defaultLakehouseId")
print("Workspace ID:", workspace_id)
print("Lakehouse ID:", lakehouse_id)

AGENT_INSTRUCTIONS = (
    "You answer questions about Zava retail inventory. Use these tables: "
    "Inventory (onHand, reorderPoint, safetyStock per product per location), "
    "Products (name, category, unitCost, leadTimeDays, supplierId), "
    "Stores (retail stores and warehouses with region/city/state), "
    "DemandHistory (weekly units sold per product per store), "
    "ExternalSignals (market signals with affectedCategories), "
    "Suppliers, and ReplenishmentOrders (past purchase/transfer orders). "
    "Always return exact numbers. Flag a product as CRITICAL when onHand is below safetyStock."
)

EXPECTED_TABLES = 7   # the 7 tables written earlier in this notebook


def _call_first(obj, names, *args, **kwargs):
    for n in names:
        fn = getattr(obj, n, None)
        if callable(fn):
            return True, fn(*args, **kwargs)
    return False, None


def _items(page):
    if isinstance(page, dict):
        return page.get("value") or page.get("elements") or page.get("items") or []
    if isinstance(page, (list, tuple)):
        return list(page)
    return []


def _attr(el, *names):
    for n in names:
        if isinstance(el, dict) and n in el:
            return el[n]
        if hasattr(el, n):
            return getattr(el, n)
    return None


def _get_elements(ds, root_id):
    """Fetch child elements of `root_id` (None = top), following pagination."""
    out = []
    token = None
    while True:
        page = None
        for kwargs in ({"root_id": root_id, "continuation_token": token},
                       {"root_id": root_id},
                       {"parent_id": root_id}):
            try:
                page = ds.get_elements(**kwargs); break
            except TypeError:
                continue
            except Exception:
                page = None; break
        if page is None:
            try:
                page = ds.get_elements()
            except Exception:
                break
        out.extend(_items(page))
        token = page.get("continuationToken") if isinstance(page, dict) else None
        if not token:
            break
    return out


def _select_all_tables(ds):
    """Walk Schemas > Schema > Tables > Table and select every TABLE leaf. Returns the count.
    Matches the EXACT leaf type 'table' (not substring -- 'Tables' is the parent container)
    and descends into any node with sub-elements. update_element is idempotent, so this is
    safe to re-run."""
    selected = 0
    stack = [None]
    seen = set()
    while stack:
        root = stack.pop()
        for el in _get_elements(ds, root):
            el_id = _attr(el, "id")
            el_type = str(_attr(el, "type") or "").strip().lower()
            has_sub = bool(_attr(el, "hasSubElements"))
            key = str(el_id)
            if el_id is None or key in seen:
                continue
            seen.add(key)
            if el_type == "table":
                try:
                    ok_sel, _ = _call_first(ds, ("update_element",), el_id, is_selected=True)
                    if not ok_sel:
                        ds.update_element(el_id, {"isSelected": True})
                    selected += 1
                except Exception as e:
                    print("update_element failed for", el_id, "->", e)
            elif has_sub:
                stack.append(el_id)   # descend into Schemas / Schema / Tables
    return selected


# Create the data agent, or re-open it if an earlier run already created it (idempotent).
try:
    agent = create_data_agent("inventory-hack-agent")
except Exception:
    agent = FabricDataAgentManagement("inventory-hack-agent")

# Public API: instructions + add the Lakehouse to staging.
_call_first(agent, ("update_settings",), ai_instructions=AGENT_INSTRUCTIONS) \
    or _call_first(agent, ("update_configuration",), instructions=AGENT_INSTRUCTIONS)
_call_first(agent, ("add_staging_datasource",),
            artifact_name_or_id=lakehouse_id, workspace_id_or_name=workspace_id) \
    or _call_first(agent, ("add_datasource",), lakehouse_id, type="lakehouse")

# After add_staging_datasource, the Lakehouse schema syncs ASYNCHRONOUSLY -- on a fresh run
# the table leaves may not be listed yet (that is why a second run "fixed" it). Poll: re-fetch
# the datasource and re-walk until all tables appear, or ~60s elapses.
selected = 0
for attempt in range(12):
    ok, ds_list = _call_first(agent, ("list_datasources", "get_datasources"))
    ds_list = list(ds_list) if ds_list else []
    if ds_list:
        selected = _select_all_tables(ds_list[0])
    if selected >= EXPECTED_TABLES:
        break
    if attempt == 0:
        print("Waiting for the Lakehouse schema to finish syncing...")
    time.sleep(5)

print(f"Selected {selected} table(s).")
if selected == 0:
    raise RuntimeError("No tables were selected (schema may still be syncing). Re-run this "
                       "cell; if it persists, tick all tables under InventoryLakehouse > dbo "
                       "in the data agent's Data tab and Publish.")

# Publish the staged config so the selection goes live on MCP / Foundry.
_call_first(agent, ("publish_staging",),
            description="Zava inventory data agent for the Agentic Inventory Planning MicroHack.") \
    or _call_first(agent, ("publish",))
print("Fabric Data Agent 'inventory-hack-agent' published.")

In [ ]:
# Smoke test -- query the PUBLISHED agent through its MCP endpoint.
# `mcp` is installed by the top cell (all %pip must be up front — it restarts the
# kernel). We resolve the agent id from the workspace, so this cell is robust even
# if run on its own, and we never fail the run on a transient MCP hiccup.
import asyncio, notebookutils
from fabric.dataagent.client import FabricDataAgentManagement
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
mgmt = FabricDataAgentManagement("inventory-hack-agent")
data_agent_id = str(mgmt._client.data_agent_id)   # resolved artifact id

mcp_url = (f"https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}"
           f"/dataagents/{data_agent_id}/agent")
QUESTION = ("How many Leaf Blower X2 units are on hand at the Portland and Seattle "
            "stores, and are they below safety stock?")

async def ask_agent(question):
    token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
    headers = {"Authorization": f"Bearer {token}"}
    async with streamablehttp_client(mcp_url, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tool = (await session.list_tools()).tools[0]
            arg = next(iter(tool.inputSchema["properties"]))
            result = await session.call_tool(tool.name, {arg: question})
            return "\n".join(b.text for b in result.content if b.type == "text")

# Fabric notebooks run inside an event loop already, so use top-level await instead
# of asyncio.run() (which errors on a running loop). Never hard-fail the headless
# run on a smoke-test error — the agent may still be fine; verify in the playground.
try:
    answer = await ask_agent(QUESTION)
    print(answer)
except Exception as e:
    print(f"[smoke-test] agent query failed ({type(e).__name__}: {e}). "
          "The agent may still be published fine — verify in the Foundry playground.")


In [ ]:
# --- Emit the IDs block: note these for the Fabric Data Agent tool in Challenge 2 ---
# Self-contained: resolve every id from the runtime + the published agent, so this
# works on its own and never depends on variables from earlier cells (a %pip restart
# earlier in the notebook clears them).
import json, notebookutils
from fabric.dataagent.client import FabricDataAgentManagement

workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
lakehouse_id = notebookutils.runtime.context.get("defaultLakehouseId")
agent_id     = str(FabricDataAgentManagement("inventory-hack-agent")._client.data_agent_id)

print("=== Your Fabric Data Agent IDs — use these in Challenge 2 ===")
print(json.dumps({
    "fabricWorkspaceId": workspace_id,
    "fabricAgentId": agent_id,
    "lakehouseId": lakehouse_id,
}, indent=2))